# 0623ver_2 老師逐字稿十點任務回應詳細報告  
## 含「檔案新增／修改過程」版本

版本標準：`0623ver_2_最終整合版_CP950_Excel直開修正版`  
報告日期：`2026-06-23`  
相對路徑基準：本報告所有路徑皆以 `0623ver_2/` 為根目錄。  

---

## 報告主旨

本 Notebook 針對老師逐字稿中整理出的十項任務，逐項說明 `0623ver_2` 如何完成。  
這份版本特別補上「不是只說結果」，而是說明：

```text
1. 為什麼要改
2. 新增了哪些檔案
3. 修改了哪些檔案
4. 檔案內大致改了什麼內容
5. 這些修改如何達成任務
6. 要怎麼驗證修改有效
```

`0623ver_2` 的定位：

```text
final_version 可實際啟動的 API / UI / DB 主系統
+ 0620 hotfix
+ 0617_B 少榆端核心資料
+ Future Prediction / Integrated Service / Monitoring / Troubleshooting 來源保留
+ Ontology / TTL / Rule 推論
+ Sequence Diagram
+ CSV Excel 直開 CP950 修正
```


# 一、十項任務與對應完成方式總覽

| 編號 | 任務 | 核心完成方式 |
|---:|---|---|
| 1 | UI → API → Service → DB → UI 串接 | 修改 API / UI Bridge / UI API client / DB schema，使 UI 可透過 API 查 PostgreSQL 並回傳畫面資料 |
| 2 | Sequence Diagram | 新增 `docs/sequence_diagram.md`，把 UI 查詢線、orchestration 線、ontology rule 線畫出來 |
| 3 | Past / Current / Future 查詢 API | 整理 time-series / summary / station-detail / component-detail endpoint 與 UI 回傳格式 |
| 4 | Future / Monitoring / Alert / Troubleshooting / DB write-back | 整合 service orchestration adapter、services runtime、DB catalog 與 future / alert 寫回表 |
| 5 | threshold 來源 CSV | 新增 / 整理 `ontology/threshold_reference.csv`，並保留 0617_B threshold template |
| 6 | CSV → TTL → Rule 推論 | 新增 `threshold_to_ttl.py`、`sprayline_threshold.ttl`、`rule_inference.py`，並修正 CSV 編碼與 parser |
| 7 | Protégé 完整 ontology | 新增 `sprayline_full_ontology.ttl` 與 0617_B modular ontology reference |
| 8 | Quality / 空壓機 / 噴幅 UI/API 問題 | 修改資料生成、dataprocess、API mapping、threshold config、UI data service |
| 9 | DB catalog / cause / response mapping | 修改 `database/setup_db.sql`，補 catalog seed 與 sensor_1hour |
| 10 | 保留 0617_B 核心資料 | 新增整理 `csv_templates/`、`knowledge/`、`schema/`、`templates/`、`service_reference_0617_B/`、`docs/0617_B_reference/` |


# 二、資料夾分工：Runtime 與 Reference

`0623ver_2` 不是單純把全部舊檔案塞在一起，而是分成「實際會跑的主系統」和「0617_B 來源追溯資料」。

## 1. 實際 runtime 主流程

這些會被 Docker / API / UI / dataprocess 實際使用：

```text
api/
database/
dataprocess/
rules/
services/
ui/
config/knowledge/
ontology/
start.ps1
docker-compose.yml
```

## 2. 0617_B 來源追溯與報告資料

這些用於保留你之前負責的少榆端設計、CSV、contract、schema、knowledge，不直接覆蓋 runtime：

```text
csv_templates/
knowledge/
schema/
templates/
docs/0617_B_reference/
service_reference_0617_B/
database_versionB_reference_0617_B/
ontology/0617_B_modular_reference/
```

## 3. 為什麼這樣整理

前一版如果把所有歷史 CSV 全部塞進來，最後版本會太亂。  
`0623ver_2` 改成只保留：

```text
1. 主系統 runtime 需要的檔案
2. 0617_B 少榆端核心資料
3. 老師逐字稿會問到的 CSV / knowledge / rule / service / DB contract / ontology
```


# 1. 把 UI → API → Service → DB → UI 的流程串起來

## 任務主旨

老師要確認 UI 不是假資料或單獨畫畫面，而是真的能透過 API 抓到 PostgreSQL 中的資料，並回傳給 UI 顯示。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `無新增單一專用檔案` | 此項主要是整合與修改既有 API / UI / DB 檔案。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `api/api_server.py` | 整理 FastAPI endpoint，讓 UI 可呼叫 dashboard、time-series、station-detail、component-detail，以及 service orchestration 相關 API。 |
| `api/shaoyu_ui_bridge.py` | 新增 / 修正 UI bridge mapping，把 DB / service 查到的 sensor rows 轉成 UI 可用 JSON，包括站點、零件與 Quality 欄位。 |
| `api/time_series_service.py` | 保留 time-series 查詢服務，支援 Past / Current / Future 的 summary 與趨勢資料查詢。 |
| `api/d_integration_adapter.py` | 補 D-stage / integrated payload 轉換與 knowledge source 對應，協助 API 統一串接少榆端資料。 |
| `ui/engineer/app/services/api_data_service.py` | 修正 Engineer UI 端 API data service，讓前端能正確呼叫 API、讀取 component-detail 與 trend data。 |
| `database/setup_db.sql` | 確認 PostgreSQL 中有 UI / API 要查的 sensor、batch、future、alert、catalog 相關資料表。 |

## 這些修改如何達成任務

這一項的修改重點，是把原本分散的資料查詢、service function 與 UI 需求串成同一條流程。  
API 端負責接收 UI 請求，UI Bridge 負責把 DB 查詢結果轉成前端 JSON，database 則提供 sensor data 與狀態資料來源。  
修正後，UI 可以透過 API 拿到 dashboard、station detail、component detail、trend chart 所需資料。

## 驗證方式

啟動系統後開：
```text
http://localhost:8011/docs
http://localhost:8013
```

PowerShell 檢查：
```powershell
docker compose ps
```

UI 檢查：
```text
Engineer UI 有資料
站點卡片有數值
零件狀態可顯示
Quality / 空壓機 / 噴幅有資料
```

## 報告時可使用的說法

> 我把 UI、API、Service、DB 串成完整資料流。UI 呼叫 FastAPI endpoint，API 再透過 UI Bridge / Time Series Service 查 PostgreSQL，最後把整理好的 JSON 回傳給 UI 顯示。


# 2. 補 Sequence Diagram，說明 UI 呼叫誰、Service 呼叫誰、最後誰回傳

## 任務主旨

老師逐字稿中特別要求 Sequence Diagram，用來清楚說明 UI、API、Service、DB、Rule 的呼叫順序。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `docs/sequence_diagram.md` | 新增 Mermaid sequence diagram，包含 UI 查詢線、Service Orchestration 線、Ontology / TTL / Rule 推論線。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `docs/sequence_diagram.md` | 後續修正 Mermaid 語法，降低特殊符號造成預覽失敗的機率，並整理成三張可截圖的時序圖。 |

## 這些修改如何達成任務

新增 `sequence_diagram.md` 後，將老師關心的流程拆成三條線。  
第一條線說明 UI 查詢資料時，如何呼叫 API、Bridge、Service、DB、Rule 並回到 UI。  
第二條線說明 integrated run once 如何觸發 Future、Monitoring、Alert、DB write-back。  
第三條線說明 threshold CSV 如何轉 TTL，再由 Rule 推論狀態。

## 驗證方式

打開：
```text
docs/sequence_diagram.md
```

用 VS Code 或 Mermaid Live 確認能看到三張圖：

```text
1. UI 查詢 Past / Current / Future 與零件狀態
2. Service Orchestration：Future / Monitoring / Alert / DB write-back
3. Ontology / TTL / Rule 推論流程
```

也可以用 Mermaid Live 匯出 PNG / SVG。

## 報告時可使用的說法

> 我補了 Sequence Diagram，把老師問的「UI 呼叫誰、再呼叫誰、最後誰回傳」畫成三條線，包含 UI 查詢線、service orchestration 線與 ontology rule 推論線。


# 3. 整合 Past / Current / Future 查詢 API

## 任務主旨

老師希望 Past / Current / Future 查詢不是散落在不同程式，而是能透過一致的 API 給 UI 使用。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `無新增單一專用檔案` | 此項主要修改既有 API / service / UI data service。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `api/api_server.py` | 保留並整理 dashboard-data、time-series summary、station-detail、component-detail 等 endpoint。 |
| `api/time_series_service.py` | 提供 time-series 查詢邏輯，支援 summary / station / component / trend 類資料。 |
| `api/shaoyu_ui_bridge.py` | 將 Past / Current / Future 查詢結果整理成 UI 所需欄位。 |
| `ui/engineer/app/services/api_data_service.py` | 前端呼叫 API 的邏輯與 component-detail / trend data mapping 對齊。 |

## 這些修改如何達成任務

這一項是把 UI 要看的時間序列資料整理成穩定 API。  
原本 UI 可能只看得到部分 current data，或 component-detail 欄位對不齊。  
修正後 API 會統一輸出 summary、station detail、component detail、trend data，讓 UI 能用同一套資料格式呈現 Past / Current / Future。

## 驗證方式

Swagger 檢查：
```text
http://localhost:8011/docs
```

確認有類似 endpoint：
```text
/api/dashboard-data
/api/time-series/ui/summary
/api/time-series/ui/station-detail
/api/time-series/ui/component-detail
```

UI 檢查：
```text
http://localhost:8013
```

確認 summary、站點 detail、零件 detail、趨勢圖可顯示。

## 報告時可使用的說法

> 我把 Past / Current / Future 查詢需求整合到 API 查詢線，UI 可以透過 dashboard-data、summary、station-detail、component-detail 取得不同時間窗與不同零件的資料。


# 4. 整合 Future Prediction / Monitoring / Alert / Troubleshooting / DB write-back

## 任務主旨

老師要確認少榆端不是只有單獨 function，而是能整合成 service orchestration，並能寫回 DB。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `service_reference_0617_B/` | 新增整理 0617_B 原始 service source reference，保留 Future / Integrated / Monitoring / Troubleshooting 的來源追溯。 |
| `database_versionB_reference_0617_B/` | 新增整理 0617_B Database/versionB reference，保留 DB 對接來源。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `api/service_orchestration_adapter.py` | 整理 API 與 integrated service 的橋接，使 API 可呼叫整合服務。 |
| `api/api_server.py` | 保留 service orchestration endpoint，提供 integrated run once / write_back 類操作。 |
| `services/future_service/` | 保留 runtime future prediction service。 |
| `services/integrated_service/` | 保留 runtime integrated service，作為 orchestration 主流程。 |
| `services/monitoring_worker/` | 保留 runtime monitoring worker，處理異常偵測。 |
| `services/troubleshooting_service/` | 保留 runtime troubleshooting service，處理原因與處置建議。 |
| `database/setup_db.sql` | 補 future_prediction_result、alert_event、catalog / link table 等 schema 與 seed。 |

## 這些修改如何達成任務

整合方式分成 runtime 與 reference 兩層。  
runtime 使用根目錄 `services/` 中的服務與 API adapter 串接。  
0617_B 原始 service source 則整理到 `service_reference_0617_B/`，保留 Future、Integrated、Monitoring、Troubleshooting 的原始設計來源，方便報告與追溯。  
DB 寫回所需的 future、alert、catalog table 則在 `database/setup_db.sql` 補齊。

## 驗證方式

Swagger 測試 integrated service：
```text
http://localhost:8011/docs
```

找：
```text
/api/service-orchestration/integrated/run-one
/api/service-orchestration/integrated/run-once
```

DB 檢查 future 寫回：
```sql
SELECT prediction_id, batch_id, station_id,
       predicted_ok_rate, predicted_ng_count,
       risk_level, created_at
FROM future_prediction_result
ORDER BY created_at DESC
LIMIT 20;
```

DB 檢查 alert：
```sql
SELECT event_id, station_id, sensor_name, state, cause, ts, message
FROM alert_event
ORDER BY ts DESC
LIMIT 20;
```

## 報告時可使用的說法

> 我把 Future Prediction、Monitoring、Alert、Troubleshooting 與 DB write-back 整合到 service orchestration 線，並保留 0617_B 原始 service reference 供老師追溯。


# 5. 說清楚 threshold 來源 CSV 在哪裡

## 任務主旨

老師追問 threshold 來源，不希望只聽到「程式裡有判斷」，而是要能指出 threshold source file。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `ontology/threshold_reference.csv` | 新增 / 整理目前 CSV → TTL → Rule 流程的主要 threshold source。 |
| `csv_templates/sensor_threshold_reference.csv` | 保留 0617_B threshold template，作為舊版來源追溯。 |
| `knowledge/legacy_thresholds_from_SprayLine_3/` | 保留舊版 filter / nozzle / process thresholds reference。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `ontology/threshold_reference.csv` | 改成 CP950 / Big5 相容並加入 sep=,，讓 Excel 直接開啟較不亂碼。 |
| `ontology/threshold_to_ttl.py` | 修正讀檔邏輯，可讀 UTF-8、UTF-8 BOM、CP950、Big5、GB18030，並跳過 sep=,。 |

## 這些修改如何達成任務

先建立 `ontology/threshold_reference.csv` 作為正式 threshold 來源。  
因為 Windows Excel 直接開 UTF-8 CSV 容易亂碼，所以最後版改成 CP950 / Big5 相容，並加入 `sep=,` 讓 Excel 可分欄。  
為了讓程式仍能讀這種 Excel 直開格式，同步修改 `threshold_to_ttl.py` 的讀檔與前處理邏輯。

## 驗證方式

直接用 Excel 雙擊打開：
```text
ontology/threshold_reference.csv
csv_templates/sensor_threshold_reference.csv
```

確認：
```text
中文不亂碼
欄位有分欄
```

再測程式仍可讀：
```powershell
python ontology\threshold_to_ttl.py --csv ontology\threshold_reference.csv --ttl ontology\sprayline_threshold.ttl
```

## 報告時可使用的說法

> threshold 的正式來源是 `ontology/threshold_reference.csv`。它可直接用 Excel 檢視，也能透過 `threshold_to_ttl.py` 轉成 TTL 給 Rule 使用。


# 6. 補 CSV → TTL → Rule 推論流程

## 任務主旨

老師要看到 threshold 能從 CSV 轉成 TTL，並由 Rule 讀 TTL 推論 normal / warning / fault。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `ontology/threshold_to_ttl.py` | 新增 CSV 轉 TTL 程式。 |
| `ontology/sprayline_threshold.ttl` | 新增 Rule inference 使用的 TTL knowledge file。 |
| `ontology/rule_inference.py` | 新增 Rule 推論程式，讀 TTL 並判斷 state / cause / response。 |
| `scripts/run_ontology_rule_test.py` | 新增一鍵 smoke test 腳本。 |
| `docs/ontology_rule_test_result.json` | 新增 rule smoke test 結果記錄。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `ontology/rule_inference.py` | 修正 TTL parser，避免小數點如 2.7、13.5 造成 block 切錯；加入空上下限不應直接判 normal 的處理。 |
| `ontology/threshold_to_ttl.py` | 加入多編碼讀取與 sep=, 跳過，支援 Excel 直開格式 CSV。 |

## 這些修改如何達成任務

此流程新增一條獨立可測試的 ontology rule pipeline。  
`threshold_reference.csv` 作為來源，`threshold_to_ttl.py` 讀取 CSV 並輸出 TTL。  
`rule_inference.py` 再讀 TTL 中的 SensorThreshold 個體，依 value 和 normal / warning / fault 門檻比較，輸出 state、cause_id、response_ids。  
期間修正過 parser，解決小數點被誤判為 TTL block 結尾的問題。

## 驗證方式

重新產生 TTL：
```powershell
python ontology\threshold_to_ttl.py --csv ontology\threshold_reference.csv --ttl ontology\sprayline_threshold.ttl
```

跑 smoke test：
```powershell
python ontology\rule_inference.py --run-smoke-test
```

重點預期：
```text
air_pressure_bar 3.2 -> normal
air_pressure_bar 4.1 -> fault
spray_width_mm 145 -> fault
film_thickness_um 17.8 -> fault
```

## 報告時可使用的說法

> 我完成 CSV → TTL → Rule 推論鏈。CSV 是 threshold source，TTL 是知識檔，rule_inference.py 會讀 TTL 推論狀態並回傳 cause_id 與 response_ids。


# 7. 補 Protégé 可展示的完整 ontology

## 任務主旨

threshold-only TTL 在 Protégé 中看起來太空，所以需要一份適合最後一週展示的 full ontology。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `ontology/sprayline_full_ontology.ttl` | 新增 Protégé 展示用 full ontology。 |
| `ontology/0617_B_modular_reference/` | 保留 0617_B 原本 modular ontology reference。 |
| `docs/full_ontology_protege_guide.md` | 新增 Protégé 展示教學。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `ontology/sprayline_full_ontology.ttl` | 建立產線、站點、元件、感測指標、門檻、狀態、原因、處置建議與推論規則之間的關係。 |

## 這些修改如何達成任務

原本 `sprayline_threshold.ttl` 主要是程式推論用，內容集中在 threshold，Protégé 圖會比較空。  
因此新增 `sprayline_full_ontology.ttl`，將 SprayLineSystem、Station、Component、SensorMetric、SensorThreshold、State、Cause、ResponseAction、InferenceRule 建成一個可視化的 ontology。  
同時保留 0617_B modular ontology，避免只剩新補的展示檔。

## 驗證方式

用 Protégé 開：
```text
ontology/sprayline_full_ontology.ttl
```

看 Classes：
```text
SprayLineSystem
Station
Component
SensorMetric
SensorThreshold
State
Cause
ResponseAction
InferenceRule
```

OntoGraf 可搜尋：
```text
SprayLine_A
AirCompressor
Threshold_air_pressure_bar
Rule_Threshold_State_Inference
```

## 報告時可使用的說法

> `sprayline_threshold.ttl` 是程式推論用，`sprayline_full_ontology.ttl` 是 Protégé 展示用。展示版補上產線、站點、元件、感測指標、門檻、狀態、原因與處理建議的完整關係。


# 8. 修正 Quality、空壓機、噴幅等 UI/API 實測問題

## 任務主旨

UI/API 實測發現 Quality 沒膜厚、空壓機 3.2 誤判、Station_2 / Station_3 噴幅誤判，這些會直接影響展示。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `database/patch_sensor_1hour.sql` | 新增 sensor_1hour 補表 patch，避免 generate_data.py 使用 sensor_1hour 時失敗。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `database/generate_data.py` | 補產生 film_thickness_um 等 Quality 相關資料，讓 sensor_1min 有膜厚值。 |
| `dataprocess/dataprocess.py` | 修正 dataprocess 寫入 / 處理膜厚、噴幅、空壓機等 sensor 欄位。 |
| `api/api_server.py` | 修正 API endpoint 與 component-detail 回傳，讓 Quality / air / spray width 可被 UI 讀到。 |
| `api/shaoyu_ui_bridge.py` | 修正 UI bridge 欄位 mapping 與 component mapping。 |
| `rules/sensor_thresholds.json` | 調整後端 threshold，空壓機 normal 2.7–3.8，噴幅 normal 70–130。 |
| `ui/engineer/config/sensor_thresholds.json` | 同步調整 UI threshold，避免前後端判斷不一致。 |
| `ui/engineer/app/services/api_data_service.py` | 修正 UI API data service 讀取 Quality / component-detail / trend data。 |

## 這些修改如何達成任務

先修資料來源，讓 generate_data 與 dataprocess 寫入 `film_thickness_um`。  
接著修 API 與 UI bridge，讓 Quality 欄位能正確回傳到前端。  
最後同步調整後端與前端 threshold JSON，避免空壓機 3.2 和正常噴幅被判異常。

## 驗證方式

DBeaver 查 Quality：
```sql
SELECT station_id,
       COUNT(*) AS total_rows,
       COUNT(film_thickness_um) AS quality_count,
       MIN(film_thickness_um) AS min_film,
       MAX(film_thickness_um) AS max_film
FROM sensor_1min
GROUP BY station_id
ORDER BY station_id;
```

查空壓機 / 噴幅：
```sql
WITH latest AS (
    SELECT DISTINCT ON (station_id)
           station_id, ts, air_pressure_bar, spray_width_mm, film_thickness_um
    FROM sensor_1min
    ORDER BY station_id, ts DESC
)
SELECT station_id, air_pressure_bar, spray_width_mm, film_thickness_um
FROM latest
ORDER BY station_id;
```

預期：
```text
film_thickness_um 有值
air_pressure_bar 3.2 左右正常
spray_width_mm 70–130 正常
```

## 報告時可使用的說法

> 我針對實測問題修正了資料生成、dataprocess、API mapping、UI mapping 與 threshold config，因此 Quality 膜厚、空壓機與噴幅都能正常顯示與判斷。


# 9. 補齊 DB catalog / response / cause mapping，避免寫回失敗

## 任務主旨

Rule / Monitoring / Troubleshooting 推論出 cause_id 和 response_ids 後，DB 必須有 catalog 對應，否則寫回 alert link 或 response link 會失敗。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `database/patch_sensor_1hour.sql` | 新增 sensor_1hour table patch。 |
| `knowledge/cause_catalog_reference.csv` | 保留 0617_B cause catalog reference。 |
| `knowledge/response_catalog_reference.csv` | 保留 0617_B response catalog reference。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `database/setup_db.sql` | 補 cause_catalog、response_catalog seed，以及 sensor_1hour 建表段落。 |
| `database/generate_data.py` | 配合 schema 產生可寫入 DB 的測試資料。 |

## 這些修改如何達成任務

在 `database/setup_db.sql` 中補齊 cause / response catalog seed，確保 Monitoring 或 Rule 推論出的 cause_id / response_id 在 DB 中存在。  
也補上 `sensor_1hour`，解決 generate_data.py 執行時找不到 relation 的問題。  
0617_B 的 cause/response reference CSV 則放到 `knowledge/`，作為資料來源追溯。

## 驗證方式

檢查 cause catalog：
```sql
SELECT cause_id FROM cause_catalog ORDER BY cause_id;
```

檢查 response catalog：
```sql
SELECT response_id FROM response_catalog ORDER BY response_id;
```

檢查 future 寫回：
```sql
SELECT prediction_id, batch_id, station_id, risk_level, created_at
FROM future_prediction_result
ORDER BY created_at DESC
LIMIT 20;
```

檢查 alert：
```sql
SELECT event_id, station_id, sensor_name, state, cause, ts
FROM alert_event
ORDER BY ts DESC
LIMIT 20;
```

## 報告時可使用的說法

> 我補齊 DB 的 cause_catalog 和 response_catalog，讓 Rule / Monitoring / Troubleshooting 產生的 cause_id 與 response_ids 都能對應到 DB，避免寫回時因 mapping 缺失失敗。


# 10. 保留 0617_B 少榆端核心資料，包括 CSV、knowledge、schema、service reference、future prediction contract

## 任務主旨

最後版本不能只剩 final_version runtime；也不能把所有歷史檔案無差別塞入。需要保留 0617_B 少榆端真正重要的核心資料。

## 新增了哪些檔案或資料夾

| 新增檔案 / 資料夾 | 新增目的 |
|---|---|
| `csv_templates/` | 新增整理 0617_B 核心 CSV templates，並補 demo row。 |
| `knowledge/` | 新增整理 0617_B cause、response、troubleshooting matrix、legacy thresholds。 |
| `schema/` | 新增整理 0617_B schema reference。 |
| `templates/` | 新增整理 0617_B JSON payload templates。 |
| `docs/0617_B_reference/` | 新增整理 0617_B contracts、notes、source reference、validation。 |
| `service_reference_0617_B/` | 新增整理 0617_B future / integrated / monitoring / troubleshooting service source。 |
| `database_versionB_reference_0617_B/` | 新增整理 0617_B Database/versionB reference。 |
| `ontology/0617_B_modular_reference/` | 新增整理 0617_B modular ontology reference。 |

## 修改了哪些檔案內容

| 修改檔案 / 資料夾 | 修改內容 |
|---|---|
| `csv_templates/*.csv` | 改成 CP950 / Big5 相容，加入 sep=,，四個 template 補 demo row。 |
| `docs/0623ver_2_核心CSV檔案位置與用途.md` | 新增核心 CSV 對照說明。 |
| `docs/0623ver_2_Runtime與Reference分工.md` | 新增 runtime 與 reference 分工說明。 |
| `README_0623ver_2_最終整合說明.md` | 新增 0623ver_2 最終整合版定位與資料夾說明。 |
| `README_0623ver_2_CP950_Excel直開修正說明.md` | 新增 CSV 編碼與 Excel 直開修正說明。 |

## 這些修改如何達成任務

這一項的重點是整理，而不是亂塞全部資料。  
我把 0617_B 中和少榆端工作直接相關的 CSV、knowledge、schema、templates、future contract、service source、DB reference、ontology reference 分類放回 0623ver_2。  
同時將 CSV 改成 Excel 直開相容格式，並替空白 template 加 demo row，讓老師可以直接打開看懂資料欄位。

## 驗證方式

檢查資料夾：
```powershell
dir csv_templates
dir knowledge
dir schema
dir templates
dir docs\0617_B_reference
dir service_reference_0617_B
dir database_versionB_reference_0617_B
dir ontology\0617_B_modular_reference
```

打開 CSV：
```text
csv_templates/sensor_1min_template.csv
csv_templates/sensor_3min_template.csv
csv_templates/alert_event_template.csv
csv_templates/batch_run_template.csv
knowledge/cause_catalog_reference.csv
knowledge/response_catalog_reference.csv
knowledge/troubleshooting_matrix_reference.csv
```

預期：
```text
中文不亂碼
欄位有分欄
template 有 demo row
```

## 報告時可使用的說法

> 我保留 0617_B 少榆端核心資料，包括 CSV templates、knowledge、schema、templates、service source reference、future prediction contract、Database versionB reference 與 modular ontology。這些是報告與追溯用，runtime 則使用根目錄的 api、database、services、ui。


# 三、最後啟動與整體驗證

## 1. 啟動 0623ver_2

```powershell
cd "C:\Users\jed92\Desktop\Project-SprayLine\0623ver_2"
powershell -ExecutionPolicy Bypass -File .\start.ps1 -Mode start -WithData
```

或雙擊：

```text
啟動_API_一鍵.bat
```

## 2. 檢查 Docker container

```powershell
docker compose ps
```

預期：

```text
sprayline_db        Up / healthy
sprayline_api       Up
sprayline_engineer  Up
sprayline_frontend  Up
dataprocess         Up
```

## 3. 開啟網址

```text
API Swagger: http://localhost:8011/docs
Engineer UI: http://localhost:8013
Manager UI: http://localhost:8012
```

## 4. 測 Ontology Rule

```powershell
python ontology\threshold_to_ttl.py --csv ontology\threshold_reference.csv --ttl ontology\sprayline_threshold.ttl
python ontology\rule_inference.py --run-smoke-test
```

## 5. Protégé 展示

```text
ontology/sprayline_full_ontology.ttl
```

## 6. Sequence Diagram

```text
docs/sequence_diagram.md
```


In [ ]:
# 可選：在 0623ver_2 根目錄下執行，檢查本報告提到的重要檔案是否存在。
from pathlib import Path

important_paths = [
    "api/api_server.py",
    "api/shaoyu_ui_bridge.py",
    "api/time_series_service.py",
    "api/service_orchestration_adapter.py",
    "database/setup_db.sql",
    "database/generate_data.py",
    "dataprocess/dataprocess.py",
    "rules/sensor_thresholds.json",
    "ui/engineer/app/services/api_data_service.py",
    "ontology/threshold_reference.csv",
    "ontology/threshold_to_ttl.py",
    "ontology/sprayline_threshold.ttl",
    "ontology/rule_inference.py",
    "ontology/sprayline_full_ontology.ttl",
    "ontology/0617_B_modular_reference",
    "docs/sequence_diagram.md",
    "docs/0623ver_2_核心CSV檔案位置與用途.md",
    "docs/0623ver_2_Runtime與Reference分工.md",
    "csv_templates/sensor_1min_template.csv",
    "csv_templates/sensor_3min_template.csv",
    "csv_templates/alert_event_template.csv",
    "csv_templates/batch_run_template.csv",
    "knowledge/cause_catalog_reference.csv",
    "knowledge/response_catalog_reference.csv",
    "docs/0617_B_reference/contracts/future_prediction_result_database_alignment_0616.csv",
    "service_reference_0617_B/integrated_service/sprayline_integrated_service.py",
    "database_versionB_reference_0617_B",
]

root = Path(".")
for rel in important_paths:
    print(f"{'OK' if (root / rel).exists() else 'MISSING'}  {rel}")


# 四、最終總結

以老師逐字稿十項要求來看，`0623ver_2` 已完成以下整理：

```text
1. 主系統能啟動 API / UI / DB / dataprocess。
2. UI 查詢線與 Service Orchestration 線已用 Sequence Diagram 說明。
3. Past / Current / Future 查詢 API 已整合。
4. Future Prediction / Monitoring / Alert / Troubleshooting / DB write-back 已納入 service orchestration。
5. threshold 來源 CSV 已明確放在 ontology/threshold_reference.csv。
6. CSV → TTL → Rule 推論可本機測試。
7. Protégé 展示用 full ontology 已補上。
8. Quality / 空壓機 / 噴幅實測問題已修正。
9. DB catalog / cause / response mapping 已補齊。
10. 0617_B 少榆端核心資料已乾淨保留，不是亂塞全部歷史檔。
```

最後報告時可以強調：

> `0623ver_2` 不是單純修改 UI 或 API，而是把 final_version runtime、0620 hotfix、0617_B 核心資料、Ontology / TTL / Rule 與 Sequence Diagram 統整成一份可啟動、可追溯、可展示的最終版。
